In [1]:
import gc
import os
from glob import glob
import polars as pl
import pandas as pd
import numpy as np

TRAIN_DIR = "data/train"

# Data preparation - Helper functions

Inspiration from: https://www.kaggle.com/code/greysky/home-credit-baseline

#### Helper class

For data preprocessing and handling

In [2]:
class Helper:
    # Sets correct datatypes for cols in df
    @staticmethod
    def set_table_dtypes(df: pl.DataFrame):
        for col in df.columns:
            if col in ["case_id", "WEEK_NUM", "num_group1", "num_group2"]:
                df = df.with_columns(pl.col(col).cast(pl.Int32))
            elif col in ["date_decision"]:
                df = df.with_columns(pl.col(col).cast(pl.Date))
            elif col[-1] in ("P", "A"):
                df = df.with_columns(pl.col(col).cast(pl.Float64))
            elif col[-1] in ("M",):
                df = df.with_columns(pl.col(col).cast(pl.String))
            elif col[-1] in ("D",):
                df = df.with_columns(pl.col(col).cast(pl.Date))

        return df

    # For each date column, calculate the number of days since the decision date
    @staticmethod
    def handle_dates(df: pl.DataFrame):
        for col in df.columns:
            if col[-1] in ("D",):
                df = df.with_columns(pl.col(col) - pl.col("date_decision"))
                df = df.with_columns(pl.col(col).dt.total_days())
                df = df.with_columns(pl.col(col).cast(pl.Float32))

        df = df.drop("date_decision", "MONTH")

        return df

#### Aggregator class
For glueing all files together -> aggregates features by case_id at different depth levels. Creates maximum values for each feature:
- Numeric features (columns ending in "P" or "A")
- Date features (columns ending in "D")
- String features (columns ending in "M")
- Other categorical features (columns ending in "T" or "L")
- Group count columns

In [3]:
class Aggregator:
    @staticmethod
    def num_expr(df):
        cols = [col for col in df.columns if col[-1] in ("P", "A")]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def date_expr(df):
        cols = [col for col in df.columns if col[-1] in ("D",)]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    # Mode is the most frequent value in a column. For categorical columns, max would not make sense.
    @staticmethod
    def str_expr(df):
        cols = [col for col in df.columns if col[-1] in ("M",)]
        expr_mode = [pl.col(col).mode().first().alias(
            f"mode_{col}") for col in cols]
        return expr_mode

    # T and L can be both categorical and numerical. For numerical columns, we can use max, for categorical columns, we can use mode again bcs for cat columns, max would not make sense.
    @staticmethod
    def other_expr(df):
        cols = [col for col in df.columns if col[-1] in ("T", "L")]
        numeric_cols = [c for c in cols if df.schema[c].is_numeric()]
        cat_cols = [c for c in cols if c not in numeric_cols]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in numeric_cols]
        expr_mode = [pl.col(col).mode().first().alias(
            f"mode_{col}") for col in cat_cols]
        return expr_max + expr_mode

    @staticmethod
    def count_expr(df):
        cols = [col for col in df.columns if "num_group" in col]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def get_exprs(df):
        exprs = Aggregator.num_expr(df) + \
            Aggregator.date_expr(df) + \
            Aggregator.str_expr(df) + \
            Aggregator.other_expr(df) + \
            Aggregator.count_expr(df)

        return exprs

#### Dataframe loading functions
- read_file: Reads a single parquet file and applies type casting and aggregation
- read_files: Reads multiple parquet files matching a glob pattern, concatenates them, and removes duplicates

In [4]:
def read_file(path, depth=None):
    df = pl.read_parquet(path)
    df = df.pipe(Helper.set_table_dtypes)

    if depth in [1, 2]:
        df = df.group_by("case_id").agg(Aggregator.get_exprs(df))

    return df


def read_files(regex_path, depth=None):
    chunks = []
    for path in glob(str(regex_path)):
        df = pl.read_parquet(path)
        df = df.pipe(Helper.set_table_dtypes)

        if depth in [1, 2]:
            df = df.group_by("case_id").agg(Aggregator.get_exprs(df))

        chunks.append(df)

    df = pl.concat(chunks, how="vertical_relaxed")
    df = df.unique(subset=["case_id"])

    return df

#### Function that glues the tables together:

In [5]:
def glue(df_base: pl.DataFrame, depth_0, depth_1, depth_2) -> pl.DataFrame:
    df_base = (
        df_base
        # Add columns for month, weekday, and year of the decision date
        .with_columns(
            month_decision=pl.col("date_decision").dt.month(),
            weekday_decision=pl.col("date_decision").dt.weekday(),
            year_decision=pl.col("date_decision").dt.year(),
        )
    )

    # Join the depth dataframes to the base dataframe
    for i, df in enumerate(depth_0 + depth_1 + depth_2):
        df_base = df_base.join(df, how="left", on="case_id", suffix=f"_{i}")

    # Handle date columns by calculating the number of days since the decision date
    df_base = df_base.pipe(Helper.handle_dates)

    return df_base

# 2. Data preparation pipeline

### 2.1 Data aggregation

In [6]:
TRAIN_PARQUET_PATH = "data/df_train_full_raw.parquet"

if os.path.exists(TRAIN_PARQUET_PATH):
    df = pl.read_parquet(TRAIN_PARQUET_PATH)
    print(f"{TRAIN_PARQUET_PATH} bestaat al, ingeladen (skipping aggregatie)")
    print("train data shape:\t", df.shape)
else:
    # FULL DATASET
    data_store = {
        "df_base": read_file(f"{TRAIN_DIR}/train_base.parquet"),
        "depth_0": [
            read_file(f"{TRAIN_DIR}/train_static_cb_0.parquet"),
            read_files(f"{TRAIN_DIR}/train_static_0_*.parquet"),
        ],
        "depth_1": [
            read_files(f"{TRAIN_DIR}/train_applprev_1_*.parquet", 1),
            read_file(f"{TRAIN_DIR}/train_tax_registry_a_1.parquet", 1),
            read_file(f"{TRAIN_DIR}/train_tax_registry_b_1.parquet", 1),
            read_file(f"{TRAIN_DIR}/train_tax_registry_c_1.parquet", 1),
            read_files(f"{TRAIN_DIR}/train_credit_bureau_a_1_*.parquet", 1),
            read_file(f"{TRAIN_DIR}/train_credit_bureau_b_1.parquet", 1),
            read_file(f"{TRAIN_DIR}/train_other_1.parquet", 1),
            read_file(f"{TRAIN_DIR}/train_person_1.parquet", 1),
            read_file(f"{TRAIN_DIR}/train_deposit_1.parquet", 1),
            read_file(f"{TRAIN_DIR}/train_debitcard_1.parquet", 1),
        ],
        "depth_2": [
            read_file(f"{TRAIN_DIR}/train_credit_bureau_b_2.parquet", 2),
            read_files(f"{TRAIN_DIR}/train_credit_bureau_a_2_*.parquet", 2),
        ]
    }

    df = glue(**data_store)
    print("train data shape:\t", df.shape)

    del data_store

    df.write_parquet(TRAIN_PARQUET_PATH)
    print(f"Saved df to {TRAIN_PARQUET_PATH}")

df = df.to_pandas()
gc.collect()

data/df_train_full_raw.parquet bestaat al, ingeladen (skipping aggregatie)
train data shape:	 (1526659, 473)


0

### 2.X Temporele dataset split

(nodig om enkel onze pipeline uit te voeren op train om geen leakage te hebben)

In [7]:
# Temporele train/test split (70/30) op basis van WEEK_NUM
cutoff_week = df["WEEK_NUM"].quantile(0.7)

train_df = df[df["WEEK_NUM"] <= cutoff_week].copy()
test_df = df[df["WEEK_NUM"] > cutoff_week].copy()

l = len(df)
del df

print(f"Cutoff week: {cutoff_week}")
print(f"Train: {train_df.shape} ({len(train_df) / l:.1%})")
print(f"Test:  {test_df.shape} ({len(test_df) / l:.1%})")
print(
    f"Train weken: {train_df['WEEK_NUM'].min()}-{train_df['WEEK_NUM'].max()}")
print(f"Test weken:  {test_df['WEEK_NUM'].min()}-{test_df['WEEK_NUM'].max()}")

Cutoff week: 52.0
Train: (1088836, 473) (71.3%)
Test:  (437823, 473) (28.7%)
Train weken: 0-52
Test weken:  53-91


### 2.2 Filter missing > 95%

In [8]:
# Drop columns with >95% missing values in the training set
missing_pct_train = train_df.isna().mean()
cols_to_drop = missing_pct_train[missing_pct_train > 0.95].index.tolist()

train_df = train_df.drop(columns=cols_to_drop)
test_df = test_df.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns with >95% missing values")
print(f"Train shape after drop: {train_df.shape}")
print(f"Test shape after drop: {test_df.shape}")

Dropped 114 columns with >95% missing values
Train shape after drop: (1088836, 359)
Test shape after drop: (437823, 359)


### 2.3 Missing indicator

In [9]:
id_cols = ["case_id", "WEEK_NUM", "target"]

In [10]:
# Add a missingness indicator for every column that has missing values, calc on train
cols_with_missing = [c for c in train_df.columns
                     if c not in id_cols and train_df[c].isna().any()]


# Train
missing_indicators = train_df[cols_with_missing].isna().astype("int8")
missing_indicators.columns = [f"{c}_missing" for c in cols_with_missing]

train_df = pd.concat([train_df, missing_indicators], axis=1)

print(f"Added {len(cols_with_missing)} missingness indicator columns")
print(f"Train shape after adding indicators: {train_df.shape}")


# Test
test_missing_indicators = test_df[cols_with_missing].isna().astype("int8")
test_missing_indicators.columns = [f"{c}_missing" for c in cols_with_missing]

test_df = pd.concat([test_df, test_missing_indicators], axis=1)

print(f"Test shape after adding indicators: {test_df.shape}")

Added 293 missingness indicator columns
Train shape after adding indicators: (1088836, 652)
Test shape after adding indicators: (437823, 652)


### 2.4 Winsorization

In [11]:
# Outlier capping (winsorization) op 1e/99e percentiel, only on train_df (geen leakage)
# Not applied to "_missing" (missingness indicators)
numeric_cols_cap = [c for c in train_df.select_dtypes(include=[np.number]).columns
                    if c not in id_cols and not c.endswith("_missing")]

lower_bounds = train_df[numeric_cols_cap].quantile(0.025)
upper_bounds = train_df[numeric_cols_cap].quantile(0.975)

train_df[numeric_cols_cap] = train_df[numeric_cols_cap].clip(
    lower=lower_bounds, upper=upper_bounds, axis=1)

test_df[numeric_cols_cap] = test_df[numeric_cols_cap].clip(
    lower=lower_bounds, upper=upper_bounds, axis=1)

print(
    f"Outlier-capping toegepast op {len(numeric_cols_cap)} numerieke kolommen (2.5e/97.5e percentiel)")

Outlier-capping toegepast op 279 numerieke kolommen (2.5e/97.5e percentiel)


### 2.5 Median imputation

In [12]:
# Mediaan imputatie voor numerieke kolommen
numeric_cols_impute = [c for c in train_df.select_dtypes(include=[np.number]).columns
                       if c not in id_cols and not c.endswith("_missing")]

train_medians = train_df[numeric_cols_impute].median()

train_df[numeric_cols_impute] = train_df[numeric_cols_impute].fillna(
    train_medians)
test_df[numeric_cols_impute] = test_df[numeric_cols_impute].fillna(
    train_medians)

print(
    f"Mediaan-imputatie toegepast op {len(numeric_cols_impute)} numerieke kolommen")
print(
    f"Resterende NaN's in numerieke kolommen: train = {train_df[numeric_cols_impute].isna().sum().sum()}, test = {test_df[numeric_cols_impute].isna().sum().sum()}")

Mediaan-imputatie toegepast op 279 numerieke kolommen
Resterende NaN's in numerieke kolommen: train = 0, test = 0


### 2.6 Categorische encoding

In [13]:
# Frequency encoding voor categorische kolommen (enkel obv train_df, geen leakage)
cat_cols = [c for c in train_df.select_dtypes(exclude=[np.number]).columns
            if c not in id_cols]

freq_maps = {}
for col in cat_cols:
    freq_maps[col] = train_df[col].value_counts()
    train_df[col] = train_df[col].map(
        freq_maps[col]).fillna(0).astype("float32")
    test_df[col] = test_df[col].map(
        freq_maps[col]).fillna(0).astype("float32")

print(f"Frequency encoding toegepast op {len(cat_cols)} categorische kolommen")
print(f"Train shape na encoding: {train_df.shape}")
print(f"Test shape na encoding: {test_df.shape}")

Frequency encoding toegepast op 77 categorische kolommen
Train shape na encoding: (1088836, 652)
Test shape na encoding: (437823, 652)


### 2.7 Standardisation

In [14]:
# Variantie-check (voor alle numerieke feature-kolommen, behalve id/target): kolommen
# met std = 0 droppen zodat we nergens door 0 delen. Eén centrale check die zowel de
# standaardisatie hieronder als het correlatiefilter (2.8) verderop afdekt.
numeric_check_cols = [c for c in train_df.select_dtypes(include=[np.number]).columns
                      if c not in id_cols]

stds = train_df[numeric_check_cols].std()
zero_std_cols = stds[stds == 0].index.tolist()

train_df = train_df.drop(columns=zero_std_cols)
test_df = test_df.drop(columns=zero_std_cols)

print(f"Dropped {len(zero_std_cols)} kolommen met std = 0: {zero_std_cols}")
print(f"Train shape na drop: {train_df.shape}")
print(f"Test shape na drop: {test_df.shape}")

Dropped 34 kolommen met std = 0: ['year_decision', 'assignmentdate_4527235D', 'responsedate_4527233D', 'actualdpdtolerance_344P', 'applicationcnt_361L', 'clientscnt3m_3712950L', 'clientscnt6m_3712949L', 'clientscnt_257L', 'clientscnt_360L', 'clientscnt_493L', 'commnoinclast6m_3546845L', 'deferredmnthsnum_166L', 'lastapprcommoditytypec_5251766M', 'lastrejectcommodtypec_5251769M', 'mastercontrelectronic_519L', 'mastercontrexist_109L', 'numnotactivated_1143L', 'numpmtchanneldd_318L', 'posfpd10lastmonth_333P', 'posfpd30lastmonth_3976960P', 'max_actualdpd_943P', 'max_recorddate_4527225D', 'max_outstandingamount_354A', 'max_overdueamount_31A', 'max_residualamount_488A', 'max_totaldebtoverduevalue_718A', 'max_totaloutstanddebtvalue_668A', 'max_numberofoutstandinstls_520L', 'max_numberofoverdueinstls_834L', 'max_periodicityofpmts_1102L', 'max_periodicityofpmts_837L', 'max_pmts_month_158T', 'max_pmts_month_706T', 'max_pmts_year_1139T']
Train shape na drop: (1088836, 618)
Test shape na drop: (43

In [15]:
# Kolommen die we niet standaardiseren: id/target, datumcomponenten van de
# beslissingsdatum, en de binaire _missing-indicatoren
standardize_cols = [c for c in train_df.select_dtypes(include=[np.number]).columns
                    if c not in id_cols + ["month_decision", "weekday_decision", "year_decision"]
                    and not c.endswith("_missing")]

# Standaardisatie (z-score), enkel obv train_df (geen leakage)
train_means = train_df[standardize_cols].mean()
train_stds = train_df[standardize_cols].std()

train_df[standardize_cols] = (
    train_df[standardize_cols] - train_means) / train_stds
test_df[standardize_cols] = (
    test_df[standardize_cols] - train_means) / train_stds

print(f"Standaardisatie toegepast op {len(standardize_cols)} kolommen")
print(f"Train shape na standaardisatie: {train_df.shape}")
print(f"Test shape na standaardisatie: {test_df.shape}")

Standaardisatie toegepast op 320 kolommen
Train shape na standaardisatie: (1088836, 618)
Test shape na standaardisatie: (437823, 618)


### 2.8 Correlation filter

In [16]:
# Drop highly correlated features
threshold = 0.7
protected_cols = id_cols

# Get only numeric columns (excluding protected cols)
# (variantie-check + drop van std=0 kolommen gebeurde al hierboven in 2.7)
numeric_cols = [col for col in train_df.select_dtypes(include=[np.number]).columns
                if col not in protected_cols]

# Geen NaN's meer op dit punt (na imputatie/encoding), dus np.corrcoef kan gebruikt worden:
# dat leunt op BLAS-matrixvermenigvuldiging en is een stuk sneller dan pandas' pairwise .corr()
values = train_df[numeric_cols].to_numpy(dtype="float32")
corr_matrix = pd.DataFrame(
    np.corrcoef(values, rowvar=False), index=numeric_cols, columns=numeric_cols
).abs()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
corr_upper = corr_matrix.where(mask)

to_drop = [col for col in corr_upper.columns if any(
    corr_upper[col] > threshold)]

train_df = train_df.drop(columns=to_drop)
test_df = test_df.drop(columns=to_drop)

print(
    f"Dropping {len(to_drop)} highly correlated columns (threshold={threshold})")
print(f"Train shape after correlation filter: {train_df.shape}")
print(f"Test shape after correlation filter: {test_df.shape}")

Dropping 412 highly correlated columns (threshold=0.7)
Train shape after correlation filter: (1088836, 206)
Test shape after correlation filter: (437823, 206)


# Save to disk + View df

In [17]:
train_df.to_parquet("data/df_train_preprocessed.parquet")
test_df.to_parquet("data/df_test_preprocessed.parquet")

In [18]:
print("Preprocessed train_df: ")
train_df

Preprocessed train_df: 


,case_id,WEEK_NUM,target,month_decision,weekday_decision,assignmentdate_238D,birthdate_574D,dateofbirth_337D,days120_123L,days30_165L,...,max_credlmt_935A_missing,max_dpdmax_757P_missing,max_outstandingamount_362A_missing,max_totaldebtoverduevalue_718A_missing,max_numberofoverdueinstlmaxdat_641D_missing,max_annualeffectiverate_199L_missing,max_annualeffectiverate_63L_missing,mode_role_1084L_missing,mode_type_25L_missing,max_amount_416A_missing
0,0,0,0,1,4,0.077125,0.091781,0.125197,-0.310926,-0.565845,...,1,1,1,1,1,1,1,0,0,1
1,1,0,0,1,4,0.077125,0.091781,0.125197,-0.310926,-0.565845,...,1,1,1,1,1,1,1,0,0,1
2,2,0,0,1,5,0.077125,0.091781,0.125197,-0.310926,-0.565845,...,1,1,1,1,1,1,1,0,0,1
3,3,0,0,1,4,0.077125,0.091781,0.125197,-0.310926,-0.565845,...,1,1,1,1,1,1,1,0,0,1
4,4,0,1,1,5,0.077125,0.091781,0.125197,-0.310926,-0.565845,...,1,1,1,1,1,1,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1479675,2656430,52,0,1,1,0.077125,0.091781,-0.147920,0.230528,-0.565845,...,1,0,0,0,1,1,1,0,0,1
1479676,2656431,52,0,1,1,0.077125,0.091781,0.656487,3.479255,2.010193,...,0,0,0,0,1,1,1,0,0,1
1479677,2656432,52,0,1,1,0.077125,0.091781,-1.736807,1.313437,-0.565845,...,1,0,0,0,1,1,0,0,0,1
1479678,2656433,52,0,1,1,0.077125,0.091781,1.051976,-0.852381,-0.565845,...,1,0,1,0,1,1,1,0,0,1


In [19]:
print("Preprocessed test_df: ")
test_df

Preprocessed test_df: 


,case_id,WEEK_NUM,target,month_decision,weekday_decision,assignmentdate_238D,birthdate_574D,dateofbirth_337D,days120_123L,days30_165L,...,max_credlmt_935A_missing,max_dpdmax_757P_missing,max_outstandingamount_362A_missing,max_totaldebtoverduevalue_718A_missing,max_numberofoverdueinstlmaxdat_641D_missing,max_annualeffectiverate_199L_missing,max_annualeffectiverate_63L_missing,mode_role_1084L_missing,mode_type_25L_missing,max_amount_416A_missing
42469,42469,53,0,1,2,0.077125,0.091781,1.335056,2.396346,2.010193,...,0,0,0,0,1,0,1,0,0,1
42496,42496,53,0,1,3,0.077125,0.091781,0.906429,0.230528,-0.565845,...,0,1,1,1,0,1,1,0,0,1
42514,42514,53,0,1,4,0.077125,0.091781,1.004976,0.230528,-0.565845,...,0,1,1,1,1,1,1,0,0,1
42521,42521,53,0,1,2,0.077125,0.091781,0.125197,-0.310926,-0.565845,...,1,1,1,1,1,1,1,0,0,1
42533,42533,53,0,1,5,0.077125,0.091781,1.545579,2.396346,3.298212,...,0,0,0,0,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1526654,2703450,91,0,10,1,0.077125,0.091781,-1.367308,-0.852381,-0.565845,...,1,0,0,0,1,1,1,0,0,1
1526655,2703451,91,0,10,1,0.077125,0.091781,-2.092444,-0.852381,-0.565845,...,1,0,0,0,1,1,1,0,0,1
1526656,2703452,91,0,10,1,0.077125,0.091781,0.023618,0.230528,-0.565845,...,0,0,1,0,0,1,1,0,0,1
1526657,2703453,91,0,10,1,0.077125,0.091781,-2.101540,0.230528,0.722174,...,1,0,0,0,1,1,1,0,0,0
